In [ ]:
import os
import sys

# Project root = parent of notebooks/
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project Root:", PROJECT_ROOT)

The Walkthrough Architecture

                 walkthrough.ipynb
                        │
─────────────────────────────────────────────────────
                        │
         Configure logging & choose model
                        │
                        ▼
               Load HuggingFace Model
                        │
                        ▼
              Wrap with jlens.from_hf()
                        │
                        ▼
            Load Pretrained Jacobian Lens
                        │
                        ▼
          Apply Jacobian Lens to a Prompt
                        │
                        ▼
      Compare Logit Lens vs Jacobian Lens
                        │
                        ▼
          Interactive Visualization
                        │
                        ▼
        Train Your Own Jacobian Lens

# Jacobian lens — walkthrough

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation. 

In [17]:
import jlens

jlens.configure_logging()

MODEL_NAME = "gpt2"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

# we will fit our own jacobian lens later.
# For now don't load any pretrained lens.

print("Model:", MODEL_NAME)

Model: gpt2


## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface

In [18]:
import torch
import transformers

device = (
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
).to(device)

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)

model = jlens.from_hf(hf_model, tokenizer)

print(model)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5048.94it/s]


HFLensModel(GPT2LMHeadModel, n_layers=12, d_model=768)


In [24]:
import os

print(os.path.exists("../english_corpus/english_prompts.txt"))

False


## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [29]:
from english_corpus.load_prompts import load_english_prompts

prompts = load_english_prompts(
    "../english_corpus/generated/english_prompts.txt",
    n_prompts =1000,
)
print(len(prompts))
print(prompts[:5])

159
['The capital of France is Paris.', 'The capital of Japan is Tokyo.', 'The capital of India is New Delhi.', 'The capital of Germany is Berlin.', 'The capital of Italy is Rome.']


The fitting cell

In [30]:
import os

os.makedirs("outputs/checkpoints", exist_ok=True)

lens = jlens.fit(
    model,
    prompts=prompts,
    source_layers=None,
    dim_batch=8,
    max_seq_len=64,
    checkpoint_path="outputs/checkpoints/gpt2_jlens.ckpt",
)

lens.save("outputs/checkpoints/english_gpt2_lens.pt")

print(lens)

[  5h23m +19229.20s] fit: n_layers=12 d_model=768, fitting 11 source layers (target=L11) on 159 prompts
[  5h23m +  0.17s]   resuming from checkpoint: 159/159 prompts processed
[  5h23m +  0.40s] fit: done, 1 prompts


JacobianLens(d_model=768, n_prompts=1, source_layers=[0..10] (11 layers))


In [ ]:
""".             JacobianLens
                  │
                  ├── n_prompts = 5
                  │
                  ├── d_model = 768
                  │
                  ├── source_layers
                  │     │
                  │     ├── 0
                  │     ├── 3
                  │     ├── 6
                  │     └── 9
                  │
                  └── jacobians
                        │
                        ├── J₀
                        │      768×768
                        │
                        ├── J₃
                        │      768×768
                        │
                        ├── J₆
                        │      768×768
                        │
                        └── J₉
                              768×768 """

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [31]:
prompt = "The largest planet in our solar system is"
layers = lens.source_layers

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-1])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-1], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  0 logit-lens: [' not', ' also', ' now', ' still', ' a']
L  0 J-lens:     [' tremend', 'ccording', 'uilt', ' livest', ' carbohyd']
L  1 logit-lens: [' now', ' not', ' also', ' currently', ' still']
L  1 J-lens:     [' tremend', ' perched', ' grows', ' eagerly', ' participates']
L  2 logit-lens: [' now', ' not', ' currently', ' still', ' also']
L  2 J-lens:     [' tremend', ' perched', ' toget', ' confir', ' mosqu']
L  3 logit-lens: [' now', ' currently', ' not', ' still', ' also']
L  3 J-lens:     [' mosqu', ' tremend', ' perched', ' trave', ' confir']
L  4 logit-lens: [' now', ' not', ' currently', ' still', ' probably']
L  4 J-lens:     [' perched', ' nesting', ' tremend', ' mosqu', ' situated']
L  5 logit-lens: [' now', ' probably', ' currently', ' still', ' undoubtedly']
L  5 J-lens:     [' tremend', ' perched', ' mathemat', ' nesting', 'quartered']
L  6 logit-lens: [' probably', ' now', ' indeed', ' still', ' definitely']
L  6 J-lens:     [' perched', ' tremend', 'quartered', ' 

## 4. Render a slice page (inline)

`compute_slice` + `build_page` produce an interactive position × layer view of the lens's token ranks (the `?` in the corner explains the controls). `mode="embed"` inlines everything so the page is self-contained.

In [ ]:
import gzip
import json

from jlens.examples import EXAMPLES, resolve_prompt
from jlens.vis import build_page, compute_slice, notebook_iframe

# English gloss for Qwen's Chinese/Japanese/Korean vocab tokens (machine-
# generated, best-effort), shown next to
# the token in the page (alt_token=).
gloss = None

example = next(e for e in EXAMPLES if e.slug == "multihop")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(
    model,
    lens,
    prompt,
    layer_stride=2,
    # Empirically on Qwen, the interesting word tokens trail punctuation and
    # single-character tokens in the raw top-K; mask to word-like tokens only.
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
)
notebook_iframe(page)

## 5. Render a slice page (served)

For longer prompts prefer `mode="fetch"`: `build_page` writes the data as sidecar files to `out_dir` and the page fetches rank files lazily on pin, so it stays small regardless of how many tokens are tracked.

In [ ]:
import os
import threading
from functools import partial
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

example = next(e for e in EXAMPLES if e.slug == "ascii-face")
prompt = resolve_prompt(example, tokenizer)

slice_data = compute_slice(model, lens, prompt, mask_display=True)
out_dir = Path("slices") / example.slug
page, _, _ = build_page(
    slice_data,
    prompt,
    title=example.section,
    description=example.description,
    alt_token=gloss,
    mode="fetch",
    out_dir=out_dir,
)
(out_dir / "index.html").write_text(page)

if "_jlens_httpd" not in globals():
    _handler = partial(SimpleHTTPRequestHandler, directory=os.path.abspath("slices"))
    _jlens_httpd = HTTPServer(("127.0.0.1", 0), _handler)
    threading.Thread(target=_jlens_httpd.serve_forever, daemon=True).start()
print(f"-> http://localhost:{_jlens_httpd.server_address[1]}/{example.slug}/")

### More to explore

A few more prompts are bundled in `jlens.examples.EXAMPLES` — change the `slug` above and see what surfaces, or try a prompt of your own.

In [ ]:
for e in EXAMPLES:
    print(f"{e.slug:>24}  {e.section}")

## 6. Fitting

`fit(model, prompts)` computes `J_l` over the supplied prompts. 100 prompts is enough for a usable lens; the released lenses use 1000. `dim_batch` is the memory knob — each prompt does `ceil(d_model / dim_batch)` backward passes on a retained graph.

In [ ]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=100)
lens = jlens.fit(
    model, prompts, dim_batch=32, max_seq_len=128, checkpoint_path="ckpt.pt"
)
lens.save("jacobian_lens.pt")